In [1]:
# --- Cell 1: Imports, environment, helpers -----------------------------------
# Purpose: Load core libraries, print versions, define lightweight logging.

import sys, os, json, textwrap, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

def log(msg=""):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] {msg}")

# Print environment: versions & basic context
log("Environment")
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", __import__("sklearn").__version__)

# Optional libraries (will be used only if present)
try:
    import umap
    HAS_UMAP = True
    log("umap-learn found")
except Exception:
    HAS_UMAP = False
    log("umap-learn NOT found (skipping UMAP)")

# You can optionally add Leiden/Louvain support:
# !pip install igraph leidenalg
try:
    import igraph as ig, leidenalg as la
    HAS_LEIDEN = True
    log("leidenalg found")
except Exception:
    HAS_LEIDEN = False
    log("leidenalg NOT found (skipping Leiden clustering)")

# Matplotlib defaults
plt.rcParams["figure.figsize"] = (6.5, 5.0)
plt.rcParams["axes.grid"] = True


/home/rohit/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


[11:47:40] Environment
Python: 3.10.12
NumPy: 1.26.4
pandas: 2.3.1
scikit-learn: 1.6.1


2025-08-25 11:47:45.084902: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-25 11:47:45.084970: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-25 11:47:45.191189: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-25 11:47:45.407202: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[11:47:46] umap-learn NOT found (skipping UMAP)
[11:47:46] leidenalg found


In [10]:
CONFIG = {
    "output_dir": "track3_outputs",

    "paths": {
        "phase_a": "PhaseA_artifacts/A12_node_features.csv",          # per-neuron causal signatures
        "phase_b": "PhaseB_artifacts/B07_participation_scores.csv",   # motif participation scores
        "phase_c": "PhaseC_artifacts/C05_attractor_fractions.csv",    # Basal/NPR10 fractions
        "annotations": "metadata/neuron_annotations.csv",             # optional classes (sensory/motor/etc.)
    },

    "canonical_id": "neuron",

    "id_columns": {
        "phase_a": "neuron",        # check exact col in A12_node_features.csv
        "phase_b": "neuron",        # check exact col in B07_participation_scores.csv
        "phase_c": "neuron",        # confirmed
        "annotations": "neuron",    # check in metadata file
    },

    "class_column_in_annotations": "class",   # if present in annotations
    "geometry_columns_in_annotations": [],

    "drop_columns_exact": [],
    "exclude_if_contains": ["neuron", "name", "id", "class", "label"],
    "feature_allowlist": [],

    "max_col_missing_frac": 0.35,
    "imputer_strategy": "median",
    "scaler": "standard",

    "pca": {"use": True, "n_components": None, "evr_target": 0.95},
    "umap": {"use": True, "n_components": 2, "n_neighbors": 15, "min_dist": 0.1},
    "tsne": {"use": False, "perplexity": 30, "n_iter": 1000},

    "knn_k": 15,
    "cluster_ks": [4, 5, 6, 7, 8, 10],
    "n_restarts_for_stability": 5,

    "enrichment_contains": ["_frac", "Basal", "NPR10", "motif", "participation"],
    "random_state": 42,
    "allow_infer_columns": False,
}


In [11]:
# --- Cell 3: Load utilities ---------------------------------------------------
# Purpose: Robust CSV/TSV detection, ID harmonization, column vetting, merging.

import csv

def read_table_auto(path: str) -> pd.DataFrame:
    """
    Load CSV/TSV with delimiter auto-detection and safe dtype handling.
    """
    if path is None:
        return None
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Path not found: {p}")
    with open(p, "r", newline="") as f:
        sample = f.read(4096)
        dialect = csv.Sniffer().sniff(sample, delimiters=",\t;|")
        delim = dialect.delimiter
    df = pd.read_csv(p, sep=delim, engine="python")
    return df

def ensure_id_column(df: pd.DataFrame, source_key: str, desired_name: str) -> pd.DataFrame:
    """
    Rename the per-source ID column to the canonical desired_name.
    Will not infer unless CONFIG['allow_infer_columns'] is True.
    """
    src_col = CONFIG["id_columns"].get(source_key, None)
    if src_col is None:
        if CONFIG["allow_infer_columns"]:
            # VERY conservative inference: look for exact matches only
            candidates = [c for c in df.columns if c.lower() in ["neuron", "neuron_id", "name", "id", "cell"]]
            if len(candidates) != 1:
                raise ValueError(
                    f"[{source_key}] Cannot infer ID column. "
                    f"Candidates found: {candidates}. Set id_columns['{source_key}'] explicitly."
                )
            src_col = candidates[0]
        else:
            raise ValueError(
                f"[{source_key}] id_columns['{source_key}'] not set. "
                f"Please specify the ID column name in CONFIG['id_columns']."
            )

    if src_col not in df.columns:
        raise KeyError(f"[{source_key}] ID column '{src_col}' not in columns: {list(df.columns)[:12]}...")

    df = df.copy()
    if src_col != desired_name:
        if desired_name in df.columns:
            raise ValueError(f"[{source_key}] desired ID name '{desired_name}' already exists in df.")
        df.rename(columns={src_col: desired_name}, inplace=True)
    # strip whitespace from IDs for safety
    df[desired_name] = df[desired_name].astype(str).str.strip()
    return df

def add_prefix(df: pd.DataFrame, prefix: str, skip_cols: list) -> pd.DataFrame:
    """
    Add a prefix to all columns except those in skip_cols.
    """
    df = df.copy()
    rename_map = {
        c: f"{prefix}{c}" for c in df.columns if c not in skip_cols
    }
    return df.rename(columns=rename_map)

def drop_constant_columns(df: pd.DataFrame) -> pd.DataFrame:
    nunique = df.nunique(dropna=True)
    keep = nunique[nunique > 1].index.tolist()
    dropped = [c for c in df.columns if c not in keep]
    if dropped:
        log(f"Dropping {len(dropped)} constant columns: {dropped[:10]}{'...' if len(dropped)>10 else ''}")
    return df[keep]

def select_feature_columns(df: pd.DataFrame) -> list:
    """
    Choose feature columns from df (post-merge) given CONFIG rules.
    """
    if CONFIG["feature_allowlist"]:
        # STRICT mode: only use allowlisted columns that exist
        features = [c for c in CONFIG["feature_allowlist"] if c in df.columns]
        log(f"Using feature allowlist → {len(features)} columns.")
        return features

    # Start with numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # Remove obvious non-features by substring
    excluded = set()
    for col in numeric_cols:
        name_l = col.lower()
        if any(tok.lower() in name_l for tok in CONFIG["exclude_if_contains"]):
            excluded.add(col)
    features = [c for c in numeric_cols if c not in excluded]

    # Also remove geometry if you set those and don't want them as features (keep them in metadata)
    for gcol in CONFIG["geometry_columns_in_annotations"]:
        if gcol in features:
            features.remove(gcol)

    log(f"Selected {len(features)} numeric feature columns (after exclusions).")
    return features


In [12]:
# --- Cell 4: Load & merge sources --------------------------------------------
# Purpose: Read Phase A/B/C + annotations, align to canonical ID, add prefixes, outer-merge.

CANON = CONFIG["canonical_id"]

dfs = {}
for key in ["phase_a", "phase_b", "phase_c", "annotations"]:
    path = CONFIG["paths"].get(key)
    if path:
        log(f"Loading [{key}] from {path}")
        df = read_table_auto(path)
        log(f"  → shape: {df.shape}, columns: {list(df.columns)[:8]}{'...' if df.shape[1]>8 else ''}")
        df = ensure_id_column(df, source_key=key, desired_name=CANON)
        # Prefix all columns except canonical ID (and class column if in annotations)
        skip = [CANON]
        if key == "annotations" and CONFIG["class_column_in_annotations"]:
            skip.append(CONFIG["class_column_in_annotations"])
        df = add_prefix(df, prefix=f"{key}__", skip_cols=skip)
        dfs[key] = df

if not dfs:
    raise RuntimeError("No sources loaded. Please set CONFIG['paths'][*] to existing files in Cell 2.")

# Outer-merge in a deterministic order
order = [k for k in ["phase_a", "phase_b", "phase_c", "annotations"] if k in dfs]
merged = dfs[order[0]]
for k in order[1:]:
    merged = merged.merge(dfs[k], on=CANON, how="outer")

log(f"Merged shape: {merged.shape}")
log(f"Coverage by source (non-null {CANON}):")
for k in order:
    log(f"  - {k}: {dfs[k][CANON].nunique()} neurons")

# Quick duplicate check
dups = merged[CANON].duplicated().sum()
if dups:
    raise ValueError(f"Duplicated {CANON} values found after merge: {dups}")

# Optionally drop known non-feature columns by explicit name
to_drop = [c for c in CONFIG["drop_columns_exact"] if c in merged.columns]
if to_drop:
    merged.drop(columns=to_drop, inplace=True)
    log(f"Dropped {len(to_drop)} explicitly excluded columns.")

log("Merged preview:")
display(merged.head(3))


[12:10:04] Loading [phase_a] from PhaseA_artifacts/A12_node_features.csv
[12:10:04]   → shape: (509, 7), columns: ['neuron', 'sig_deg_out', 'sig_deg_in', 'sig_deg_total', 'gold_deg_out', 'gold_deg_in', 'gold_deg_total']
[12:10:04] Loading [phase_b] from PhaseB_artifacts/B07_participation_scores.csv


FileNotFoundError: Path not found: PhaseB_artifacts/B07_participation_scores.csv

In [ ]:
# --- Cell 5: Integrity & missingness -----------------------------------------
# Purpose: Summarize basic integrity, missingness per column, and per-source overlap.

log("Integrity report")
print("Rows (neurons):", merged.shape[0])
print("Columns (total):", merged.shape[1])

# Missingness per column (top 20)
miss = merged.isna().mean().sort_values(ascending=False)
log("Top 20 most-missing columns:")
display(miss.head(20).to_frame("missing_frac").style.format("{:.2f}"))

# Overlaps among sources (pairwise)
from itertools import combinations
id_sets = {k: set(dfs[k][CANON].dropna().astype(str)) for k in dfs}
for a, b in combinations(order, 2):
    inter = len(id_sets[a] & id_sets[b])
    log(f"Overlap[{a} ∩ {b}] = {inter}")

# Geometry columns present?
if CONFIG["geometry_columns_in_annotations"]:
    gcols = [c for c in CONFIG["geometry_columns_in_annotations"] if c in merged.columns]
    log(f"Geometry columns present: {gcols}")


In [ ]:
# --- Cell 6: Feature matrix X + preprocessing --------------------------------
# Purpose: Build the feature matrix X and a metadata df (ID, class). Handle missingness and scaling.

# Metadata (ID + optional class)
meta_cols = [CANON]
class_col = CONFIG["class_column_in_annotations"]
if class_col and class_col in merged.columns:
    meta_cols.append(class_col)

meta = merged[meta_cols].copy()

# Feature selection
feature_cols = select_feature_columns(merged)
X_raw = merged[feature_cols].copy()

# Drop features with too much missingness
col_missing = X_raw.isna().mean()
keep_cols = col_missing[col_missing <= CONFIG["max_col_missing_frac"]].index.tolist()
dropped_high_miss = [c for c in feature_cols if c not in keep_cols]
if dropped_high_miss:
    log(f"Dropping {len(dropped_high_miss)} features with missing_frac > {CONFIG['max_col_missing_frac']}")
X = X_raw[keep_cols]

# Impute
imputer = SimpleImputer(strategy=CONFIG["imputer_strategy"])
X_imputed = imputer.fit_transform(X)

# Scale
scaler_name = CONFIG["scaler"].lower()
if scaler_name == "standard":
    scaler = StandardScaler()
elif scaler_name == "robust":
    scaler = RobustScaler()
elif scaler_name == "quantile":
    scaler = QuantileTransformer(output_distribution="normal", random_state=CONFIG["random_state"])
else:
    raise ValueError(f"Unknown scaler: {CONFIG['scaler']}")

X_scaled = scaler.fit_transform(X_imputed)

log(f"Feature matrix ready: X_scaled shape = {X_scaled.shape} (from {len(feature_cols)} → {X_scaled.shape[1]} usable)")


In [ ]:
# --- Cell 7: Dimensionality reduction ----------------------------------------
# Purpose: PCA to denoise / compress; optional 2D embeddings for visualization.

np.random.seed(CONFIG["random_state"])

X_for_graph = X_scaled  # default (no PCA)
pca_model = None
pca_evr = None

if CONFIG["pca"]["use"]:
    pca_model = PCA(
        n_components=CONFIG["pca"]["n_components"],
        random_state=CONFIG["random_state"]
    )
    X_pca = pca_model.fit_transform(X_scaled)
    if CONFIG["pca"]["n_components"] is None:
        evr_cum = np.cumsum(pca_model.explained_variance_ratio_)
        n_keep = int(np.searchsorted(evr_cum, CONFIG["pca"]["evr_target"]) + 1)
        pca_model = PCA(n_components=n_keep, random_state=CONFIG["random_state"])
        X_pca = pca_model.fit_transform(X_scaled)
        log(f"PCA retaining {CONFIG['pca']['evr_target']*100:.0f}% EVR → n_components = {n_keep}")
    else:
        log(f"PCA with n_components = {CONFIG['pca']['n_components']}")
    pca_evr = pca_model.explained_variance_ratio_
    X_for_graph = X_pca

# Save PCA EVR plot (if available)
if pca_evr is not None:
    evr_cum = np.cumsum(pca_evr)
    plt.figure()
    plt.plot(evr_cum, linewidth=2)
    plt.xlabel("Number of components")
    plt.ylabel("Cumulative explained variance")
    plt.title("PCA cumulative EVR")
    plt.tight_layout()
    plt.show()

# Optional UMAP
UMAP_2D = None
if CONFIG["umap"]["use"] and HAS_UMAP:
    reducer = umap.UMAP(
        n_components=CONFIG["umap"]["n_components"],
        n_neighbors=CONFIG["umap"]["n_neighbors"],
        min_dist=CONFIG["umap"]["min_dist"],
        random_state=CONFIG["random_state"]
    )
    UMAP_2D = reducer.fit_transform(X_for_graph)
    log("UMAP 2D embedding computed.")

# Optional t-SNE
TSNE_2D = None
if CONFIG["tsne"]["use"]:
    from sklearn.manifold import TSNE
    tsne = TSNE(
        n_components=2,
        perplexity=CONFIG["tsne"]["perplexity"],
        n_iter=CONFIG["tsne"]["n_iter"],
        random_state=CONFIG["random_state"],
        init="pca",
        learning_rate="auto"
    )
    TSNE_2D = tsne.fit_transform(X_for_graph)
    log("t-SNE 2D embedding computed.")


In [ ]:
# --- Cell 8: kNN graph -------------------------------------------------------
# Purpose: Build a kNN graph on the reduced space (PCA if enabled), for graph-based clustering.

k = CONFIG["knn_k"]
nbrs = NearestNeighbors(n_neighbors=k, algorithm="auto", metric="euclidean")
nbrs.fit(X_for_graph)
distances, indices = nbrs.kneighbors(X_for_graph)

# Build a simple adjacency matrix in CSR form
from scipy.sparse import lil_matrix
n = X_for_graph.shape[0]
A = lil_matrix((n, n))
for i in range(n):
    for j in indices[i]:
        if i != j:
            A[i, j] = 1
            A[j, i] = 1  # symmetrize
A = A.tocsr()

log(f"kNN graph built: n={n}, k={k}, edges≈{A.nnz//2}")


In [ ]:
# --- Cell 9: Clustering sweep -------------------------------------------------
# Purpose: For a grid of K, run multiple clustering methods and score them.

def run_kmeans(X, K, seed):
    model = KMeans(n_clusters=K, n_init="auto", random_state=seed)
    labels = model.fit_predict(X)
    return labels, model

def run_spectral(A, K, seed):
    # SpectralClustering consumes dense affinity or precomputed; use A as connectivity
    model = SpectralClustering(
        n_clusters=K,
        affinity="precomputed",
        random_state=seed,
        assign_labels="kmeans"
    )
    labels = model.fit_predict(A)
    return labels, model

results = []
rng = np.random.RandomState(CONFIG["random_state"])

for K in CONFIG["cluster_ks"]:
    # KMeans on PCA (or scaled if PCA disabled)
    labels_km, _ = run_kmeans(X_for_graph, K, seed=rng.randint(0, 10_000))
    sil_km = silhouette_score(X_for_graph, labels_km)
    ch_km = calinski_harabasz_score(X_for_graph, labels_km)
    db_km = davies_bouldin_score(X_for_graph, labels_km)
    results.append({"method":"kmeans","K":K,"silhouette":sil_km,"ch":ch_km,"db":db_km})

    # Spectral on graph A
    try:
        labels_sp, _ = run_spectral(A, K, seed=rng.randint(0, 10_000))
        sil_sp = silhouette_score(X_for_graph, labels_sp)
        ch_sp = calinski_harabasz_score(X_for_graph, labels_sp)
        db_sp = davies_bouldin_score(X_for_graph, labels_sp)
        results.append({"method":"spectral","K":K,"silhouette":sil_sp,"ch":ch_sp,"db":db_sp})
    except Exception as e:
        log(f"[spectral K={K}] failed: {e}")

scores_df = pd.DataFrame(results).sort_values(["method","K"]).reset_index(drop=True)
log("Clustering sweep complete. Higher silhouette & CH are better; lower DB is better.")
display(scores_df)


In [ ]:
# --- Cell 10: Stability analysis ---------------------------------------------
# Purpose: Estimate label stability via pairwise ARI across multiple random seeds.

def stability_ari(X, K, method="kmeans", repeats=5, A=None, base_seed=42):
    rng = np.random.RandomState(base_seed)
    label_runs = []
    for r in range(repeats):
        seed = rng.randint(0, 10_000)
        if method == "kmeans":
            labels, _ = run_kmeans(X, K, seed=seed)
        elif method == "spectral":
            if A is None:
                raise ValueError("A (graph) required for spectral stability.")
            labels, _ = run_spectral(A, K, seed=seed)
        else:
            raise ValueError(method)
        label_runs.append(labels)

    # pairwise ARI
    aris = []
    for i in range(len(label_runs)):
        for j in range(i+1, len(label_runs)):
            aris.append(adjusted_rand_score(label_runs[i], label_runs[j]))
    return float(np.mean(aris)), float(np.std(aris))

# Pick top-2 by silhouette per method to inspect stability
stability_rows = []
for meth in scores_df["method"].unique():
    subset = scores_df[scores_df["method"]==meth].sort_values("silhouette", ascending=False).head(2)
    for _, row in subset.iterrows():
        K = int(row["K"])
        mean_ari, std_ari = stability_ari(
            X_for_graph, K, method=meth, repeats=CONFIG["n_restarts_for_stability"], A=A
        )
        stability_rows.append({
            "method": meth, "K": K,
            "silhouette": row["silhouette"],
            "stability_ARI_mean": mean_ari,
            "stability_ARI_std": std_ari
        })
stability_df = pd.DataFrame(stability_rows).sort_values(["method","silhouette"], ascending=[True,False])
log("Stability (higher ARI mean is better):")
display(stability_df)


In [ ]:
# --- Cell 11: Final clustering choice ----------------------------------------
# Purpose: Explicitly select the final method and K. No implicit assumptions.

FINAL_METHOD = "kmeans"   # "kmeans" or "spectral"
FINAL_K = 6               # set explicitly after reviewing Cells 9–10

if FINAL_METHOD == "kmeans":
    final_labels, final_model = run_kmeans(X_for_graph, FINAL_K, seed=CONFIG["random_state"])
elif FINAL_METHOD == "spectral":
    final_labels, final_model = run_spectral(A, FINAL_K, seed=CONFIG["random_state"])
else:
    raise ValueError("FINAL_METHOD must be 'kmeans' or 'spectral'.")

meta["cluster"] = final_labels
log(f"Assigned final cluster labels: method={FINAL_METHOD}, K={FINAL_K}")
display(meta.head())


In [ ]:
# --- Cell 12: Enrichment summaries -------------------------------------------
# Purpose: Summarize cluster composition and per-feature means for selected enrichment families.

# Bring back original merged for per-cluster aggregations
out = merged.copy()
out["cluster"] = meta["cluster"].values

# Cluster sizes
cluster_sizes = out.groupby("cluster").size().to_frame("n")
log("Cluster sizes:")
display(cluster_sizes)

# Composition by class (if available)
if CONFIG["class_column_in_annotations"] and CONFIG["class_column_in_annotations"] in out.columns:
    class_col = CONFIG["class_column_in_annotations"]
    comp = (out.groupby(["cluster", class_col]).size()
              .unstack(fill_value=0))
    comp_frac = comp.div(comp.sum(axis=1), axis=0)
    log("Cluster → class counts:")
    display(comp)
    log("Cluster → class fractions:")
    display(comp_frac)

# Enrichment for selected families (columns containing substrings)
enrich_blocks = {}
contains_tokens = CONFIG["enrichment_contains"]
if contains_tokens:
    cols = [c for c in out.columns if any(tok.lower() in c.lower() for tok in contains_tokens)]
    if cols:
        enrich = out.groupby("cluster")[cols].mean().round(4)
        enrich_blocks["feature_means"] = enrich
        log("Cluster → mean of selected enrichment columns:")
        display(enrich)
    else:
        log("No columns matched enrichment tokens (edit CONFIG['enrichment_contains']).")

# Save enrichment tables
save_dir = Path(CONFIG["output_dir"])
cluster_sizes.to_csv(save_dir / "cluster_sizes.csv", index=True)
if CONFIG["class_column_in_annotations"] and CONFIG["class_column_in_annotations"] in out.columns:
    comp.to_csv(save_dir / "cluster_class_counts.csv", index=True)
    comp_frac.to_csv(save_dir / "cluster_class_fractions.csv", index=True)
if "feature_means" in enrich_blocks:
    enrich_blocks["feature_means"].to_csv(save_dir / "cluster_enrichment_means.csv", index=True)


In [ ]:
# --- Cell 13: Embedding plots -------------------------------------------------
# Purpose: Quick qualitative checks of cluster separability in 2D embeddings.

def scatter2d(emb, labels, title):
    plt.figure()
    plt.scatter(emb[:,0], emb[:,1], s=16, c=labels, cmap="tab10", alpha=0.9)
    plt.title(title)
    plt.xlabel("dim-1"); plt.ylabel("dim-2")
    plt.tight_layout()
    plt.show()

# PCA (first two components of whatever space we used)
if CONFIG["pca"]["use"] and pca_model is not None and pca_model.n_components_ >= 2:
    pca2 = X_for_graph[:, :2] if X_for_graph.shape[1] >= 2 else None
    if pca2 is not None:
        scatter2d(pca2, meta["cluster"].values, f"PCA(2) — {FINAL_METHOD}, K={FINAL_K}")

# UMAP if computed
if UMAP_2D is not None:
    scatter2d(UMAP_2D, meta["cluster"].values, f"UMAP(2) — {FINAL_METHOD}, K={FINAL_K}")

# t-SNE if computed
if TSNE_2D is not None:
    scatter2d(TSNE_2D, meta["cluster"].values, f"t-SNE(2) — {FINAL_METHOD}, K={FINAL_K}")


In [ ]:
# --- Cell 14: Functional atlas export ----------------------------------------
# Purpose: Persist all key artifacts (labels, summary, embeddings, config snapshot).

save_dir = Path(CONFIG["output_dir"])
save_dir.mkdir(parents=True, exist_ok=True)

# Neuron → cluster labels
labels_df = meta[[CONFIG["canonical_id"], "cluster"]].copy()
labels_path = save_dir / "neuron_cluster_labels.csv"
labels_df.to_csv(labels_path, index=False)

# Embeddings (PCA space if used; else scaled feature space)
if CONFIG["pca"]["use"] and pca_model is not None:
    emb = X_for_graph
    emb_cols = [f"pc{i+1}" for i in range(emb.shape[1])]
    emb_df = pd.DataFrame(emb, columns=emb_cols)
    emb_df.insert(0, CONFIG["canonical_id"], meta[CONFIG["canonical_id"]].values)
    emb_df.to_csv(save_dir / "embeddings_pca.csv", index=False)

# Clustering scores & stability (from earlier cells)
scores_df.to_csv(save_dir / "clustering_scores.csv", index=False)
stability_df.to_csv(save_dir / "clustering_stability.csv", index=False)

# Config snapshot
with open(save_dir / "config_snapshot.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

log("Functional atlas artifacts saved:")
print(" -", labels_path)
print(" -", save_dir / "embeddings_pca.csv")
print(" -", save_dir / "clustering_scores.csv")
print(" -", save_dir / "clustering_stability.csv")
print(" -", save_dir / "cluster_sizes.csv")
print(" -", save_dir / "cluster_enrichment_means.csv")


In [ ]:
# --- Cell 15: OPTIONAL Leiden clustering -------------------------------------
# Purpose: If 'leidenalg' is available, run Leiden on the kNN graph and compare.

if not HAS_LEIDEN:
    log("Leiden not available; skip. To enable: pip install igraph leidenalg")
else:
    # Build igraph from adjacency
    sources, targets = A.nonzero()
    edges = list(zip(sources.tolist(), targets.tolist()))
    g = ig.Graph(n=A.shape[0], edges=edges, directed=False)
    # Leiden with fixed resolution parameter
    part = la.find_partition(g, la.RBConfigurationVertexPartition, resolution_parameter=1.0, seed=CONFIG["random_state"])
    leiden_labels = np.array(part.membership)
    meta["leiden_cluster"] = leiden_labels

    # Score vs PCA space (silhouette etc.)
    sil = silhouette_score(X_for_graph, leiden_labels)
    ch = calinski_harabasz_score(X_for_graph, leiden_labels)
    db = davies_bouldin_score(X_for_graph, leiden_labels)

    log(f"Leiden: n_clusters={len(np.unique(leiden_labels))} | silhouette={sil:.3f} | CH={ch:.1f} | DB={db:.3f}")

    meta[["leiden_cluster"]].to_csv(Path(CONFIG["output_dir"]) / "neuron_leiden_labels.csv", index=False)
    log("Saved Leiden labels.")


In [ ]:
# --- Cell 16: README.md for outputs ------------------------------------------
# Purpose: Emit a human-friendly README summarizing the pipeline & files.

readme = f"""# Track 3 — Functional Clustering / Atlas

**Generated**: {datetime.now().isoformat(timespec='seconds')}

## What this folder contains

- `neuron_cluster_labels.csv` — final cluster labels for each neuron (method={FINAL_METHOD}, K={FINAL_K}).
- `embeddings_pca.csv` — per-neuron PCA embedding used for clustering/visualization (if PCA was used).
- `clustering_scores.csv` — sweep of (method, K) with silhouette, CH, DB scores.
- `clustering_stability.csv` — mean±std ARI across repeated runs (stability proxy).
- `cluster_sizes.csv` — size of each cluster.
- `cluster_class_counts.csv` / `cluster_class_fractions.csv` — if a class column was provided.
- `cluster_enrichment_means.csv` — within-cluster means for columns matching {CONFIG["enrichment_contains"]}.
- `config_snapshot.json` — the exact CONFIG used to generate this atlas.

## How to reproduce

1. Edit `CONFIG` in Cell 2 (paths, ID columns, class/geometry, and hyperparams).
2. Run Cells 4 → 14 in order.
3. Inspect `clustering_scores.csv` and `clustering_stability.csv` to choose FINAL_METHOD and FINAL_K.
4. Re-run Cell 11 with your chosen settings, then Cells 12 → 14 to export the atlas.

## Notes

- No assumptions were made about file schemas beyond the explicit ID mappings you provide in `CONFIG`.
- PCA/UMAP/t-SNE are used only for denoising and visualization; clustering is run on PCA space if enabled.
- Stability is measured via ARI across repeated runs with different seeds (unsupervised proxy).
"""

readme_path = Path(CONFIG["output_dir"]) / "README.md"
with open(readme_path, "w") as f:
    f.write(readme)
log(f"Wrote {readme_path}")
